In [7]:
from recipes_agent.tracing import setup_tracing, flush

# Loads .env and attaches the Langfuse handler to every LangChain run in this
# kernel, so no callbacks need to be passed to the .invoke() calls below.
setup_tracing()

Langfuse tracing ON -> http://localhost:3000


In [8]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [9]:
system_prompt = """
You are a personal chef and recipe finder.

The user will provide a list of ingredients they currently have at home. Your job is to help them find recipes they can make using those ingredients.

MANDATORY RULE — WEB SEARCH REQUIRED:
You MUST use the web search tool for every single recipe-related response, with no exceptions. Never answer from memory or general knowledge alone. If you have not called the web search tool in this turn, you are not allowed to output recipe suggestions or instructions yet — search first, then respond. This applies to the first recipe suggestions AND to any follow-up request for full instructions (search again specifically for that recipe before giving steps).

For every recipe request:
1. ALWAYS use the web search tool first to find real recipes that match the user's ingredients. Do not skip this step under any circumstances, even if you think you already know a good recipe.
2. Prioritize recipes that use as many of the user's ingredients as possible.
3. Prefer recipes that require few additional ingredients.
4. Clearly mention if a recipe requires ingredients that the user did not list.
5. Return 3-5 recipe suggestions unless there are fewer suitable options.
6. Keep the suggestions concise: give the recipe name, a short description, approximate cooking time, and why it matches their ingredients.
7. Do not provide full cooking instructions unless the user explicitly asks for a specific recipe or asks for the instructions.
8. If the user asks for the full recipe, you MUST search the web again for that specific recipe before providing clear step-by-step instructions.
9. Never claim that a recipe was found through web search if you did not actually search for it.
10. Do not assume the user has ingredients they did not mention. Common pantry staples such as salt, pepper, oil, or water may be treated as optional unless the recipe depends heavily on them.
11. If for any reason a web search returns no useful results, explicitly tell the user that the search did not return good matches, rather than inventing a recipe from memory.

When the user's message contains only a list of ingredients, interpret it as a request for recipe ideas.

Example:

User:
I have chicken breast, spinach, garlic, and heavy cream.

Assistant:
[searches the web for recipes using chicken breast, spinach, garlic, and heavy cream]

Based on what you have, you could make:

1. **Creamy Tuscan Chicken** — chicken with spinach and garlic in a creamy sauce. ~30 min. Uses almost everything you have.
2. **Chicken Florentine** — pan-seared chicken with spinach in a creamy garlic sauce. ~30 min. You may need a few extra pantry ingredients.
3. **Creamy Garlic Chicken & Spinach** — a simple one-pan chicken dish with a creamy garlic sauce. ~25 min.

Want the full recipe for one of these?

---

Example:

User:
leftover rice, eggs, soy sauce, green onions

Assistant:
[searches the web for matching recipes]

Based on what you have:

1. **Egg Fried Rice** — quick fried rice with eggs, soy sauce, and green onions. ~15 min.
2. **Green Onion Fried Rice** — simple fried rice focused on eggs, scallions, and soy sauce. ~15-20 min.
3. **Soy Sauce Egg Rice** — a very simple option if you want something quick. ~10 min.

Want the full recipe for one of them?

---

Example:

User:
I only have canned beans, tortillas, and cheese.

Assistant:
[searches the web for matching recipes]

You can make:

1. **Bean & Cheese Quesadillas** — crispy tortillas filled with beans and melted cheese. ~15 min.
2. **Bean Tacos** — warm tortillas with seasoned beans and cheese. ~15 min.
3. **Bean & Cheese Burritos** — beans and cheese wrapped in tortillas for a more filling meal. ~20 min.

Want the full recipe for one of these?

---

If the user's ingredients are insufficient for a typical recipe, still try to find simple recipes that use what they have. Be transparent about any important missing ingredients.

If the user asks a follow-up such as "the first one", "give me the recipe", or "how do I make it?", understand which previously suggested recipe they are referring to, search the web for that specific recipe, and then provide its full instructions.

REMINDER: Every response involving recipes requires an actual web search call before you write your answer. This is not optional.

Now respond to the user's next message according to these rules.
"""

In [10]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("claude-haiku-4-5", temperature=0.3)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [5]:
from ipywidgets import FileUpload
from IPython.display import display
import base64

uploader = FileUpload(
    accept='.jpeg,.jpg,.png,.gif,.bmp,.webp,.tiff,.tif,.svg,.heic,.heif,.avif,image/*',
    multiple=False
)

display(uploader)

FileUpload(value=(), accept='.jpeg,.jpg,.png,.gif,.bmp,.webp,.tiff,.tif,.svg,.heic,.heif,.avif,image/*', descr…

In [11]:
import base64

uploaded_file = list(uploader.value)[0]

content_mv = uploaded_file["content"]
img_bytes = bytes(content_mv)

img_base64 = base64.b64encode(img_bytes).decode("utf-8")


def detect_mime_type(img_bytes: bytes) -> str:
    if img_bytes.startswith(b"\xff\xd8\xff"):
        return "image/jpeg"
    elif img_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return "image/png"
    elif img_bytes.startswith(b"GIF87a") or img_bytes.startswith(b"GIF89a"):
        return "image/gif"
    elif img_bytes[0:4] == b"RIFF" and img_bytes[8:12] == b"WEBP":
        return "image/webp"
    else:
        raise ValueError("Unsupported or unrecognized image format")


mime_type = detect_mime_type(img_bytes)
img_base64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Give me some recipes based on what I have"},
    {"type": "image", "base64": img_base64, "mime_type": mime_type}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

In [13]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

Based on what I can see in your fridge, here are some recipe ideas:

1. **Curry Chicken Salad with Eggs and Grapes** — A delicious salad combining hard-boiled eggs, chicken (the meat you have), and grapes with a creamy curry mayo dressing. ~20 min. Uses most of your visible ingredients.

2. **Puffy Omelette with Roasted Grapes** — A fluffy omelette with roasted grapes for a sweet and savory combination. ~25 min. Great way to use eggs and grapes together.

3. **Beefed-up Shakshuka** — Eggs poached in a flavorful meat and tomato sauce. ~40 min. Uses your eggs and meat (you may need tomatoes or tomato sauce).

4. **Stuffed Grape Leaves with Egg-Lemon Sauce** — A Mediterranean dish with meat-stuffed grape leaves and an egg-based sauce. ~45 min. Uses eggs, meat, and grapes in a traditional way.

However, I'm having trouble seeing all the details in your fridge clearly. Could you tell me more specifically what proteins and other ingredients you have? For example:
- What type of meat is in the container?
- Do you have any vegetables, dairy, or pantry staples?
- Any other items I might have missed?

This will help me give you more tailored recipe suggestions!

In [14]:
# Traces are batched — push them out before the kernel goes idle.
flush()

Media upload error: Failed to upload media due to unexpected error. Queue item marked as done. Error: [Errno 11001] getaddrinfo failed
Media upload error: Failed to upload media due to unexpected error. Queue item marked as done. Error: [Errno 11001] getaddrinfo failed
